In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '_sig_analysis.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

# Significance Tests for Table 1

Paired t-tests across 10 splits to determine:
1. Which results should be **bold** (best overall per column, significantly better than all others)
2. Which results should be **underlined** (best zero-shot per column)
3. TabPFN vs TabICL backend comparison for each method variant

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

In [ ]:
# ── Load results ──────────────────────────────────────────────────────────────
with open('../results/results.json') as f:
    res = json.load(f)

with open('../results/results_bj.json') as f:
    bj = json.load(f)

DATASETS = ['whas500', 'gbsg', 'metabric', 'support', 'flchain']

# Build a dict: method -> dataset -> [ci_split0, ..., ci_split9]
#                                     [ibs_split0, ..., ibs_split9]
def extract(res, key, metric):
    return [x[metric] for x in res[key]]

def bj_last(bj_key, ds, metric):
    """Last iteration value per split for FSA-BJ."""
    return [splits[-1][metric.upper()] for splits in bj[bj_key][ds]]

# Assemble all per-split scores
# Keys: (method_label, dataset) -> {'ci': [...], 'ibs': [...]}
DATA = {}

classical = {
    'Cox PH':        'cox',
    'Weibull AFT':   'weibull',
    'Log-Normal AFT':'lognormal',
    'RSF':           'rsf',
}
tabpfn = {
    'TabSA-CCA (PFN)': 'fsa_tabpfn',
    'TabSA-Bin (PFN)': 'bin_fsa_tabpfn',
    'TabSA-PO (PFN)':  'pseudo_fsa_tabpfn',
}
tabicl = {
    'TabSA-CCA (ICL)': 'fsa_tabicl',
    'TabSA-Bin (ICL)': 'bin_fsa_tabicl',
    'TabSA-PO (ICL)':  'pseudo_fsa_tabicl',
}

for ds in DATASETS:
    for label, key in {**classical, **tabpfn, **tabicl}.items():
        if ds in res and key in res[ds]:
            DATA[(label, ds)] = {
                'ci':  extract(res[ds], key, 'ci'),
                'ibs': extract(res[ds], key, 'ibs'),
            }

    # FSA-BJ from results_bj.json
    for bj_key, label in [('tabpfn_seed0', 'TabSA-BJ (PFN)'), ('tabicl_seed0', 'TabSA-BJ (ICL)')]:
        if bj_key in bj and ds in bj[bj_key]:
            DATA[(label, ds)] = {
                'ci':  bj_last(bj_key, ds, 'ci'),
                'ibs': bj_last(bj_key, ds, 'ibs'),
            }

ALL_METHODS = list(classical) + ['TabSA-CCA (PFN)', 'TabSA-Bin (PFN)', 'TabSA-PO (PFN)', 'TabSA-BJ (PFN)',
                                  'TabSA-CCA (ICL)', 'TabSA-Bin (ICL)', 'TabSA-PO (ICL)', 'TabSA-BJ (ICL)']
ZERO_SHOT  = [m for m in ALL_METHODS if m not in classical]

print(f"Loaded {len(DATA)} (method, dataset) pairs")

In [ ]:
# ── Summary table (mean ± std) ────────────────────────────────────────────────
rows = []
for m in ALL_METHODS:
    for ds in DATASETS:
        if (m, ds) not in DATA:
            continue
        ci  = DATA[(m, ds)]['ci']
        ibs = DATA[(m, ds)]['ibs']
        rows.append({'Method': m, 'Dataset': ds,
                     'CI_mean': np.mean(ci),  'CI_std': np.std(ci),
                     'IBS_mean': np.mean(ibs), 'IBS_std': np.std(ibs)})

summary = pd.DataFrame(rows)
pivot_ci  = summary.pivot(index='Method', columns='Dataset', values='CI_mean').reindex(ALL_METHODS)[DATASETS]
pivot_ibs = summary.pivot(index='Method', columns='Dataset', values='IBS_mean').reindex(ALL_METHODS)[DATASETS]

print("=== Mean C-index ===")
print(pivot_ci.round(3).to_string())
print("\n=== Mean IBS ===")
print(pivot_ibs.round(3).to_string())

In [ ]:
# ── Significance testing helpers ──────────────────────────────────────────────
ALPHA = 0.05

def paired_ttest(a, b):
    """Two-sided paired t-test. Returns p-value."""
    return stats.ttest_rel(a, b).pvalue

def best_methods(methods, ds, metric, higher_is_better=True):
    """
    Return the set of methods that are:
      - Best mean on (ds, metric)
      - Not significantly worse than the overall best (p >= ALPHA)
    I.e. the 'statistical champion set'.
    """
    available = [m for m in methods if (m, ds) in DATA]
    if not available:
        return set()

    scores = {m: np.mean(DATA[(m, ds)][metric]) for m in available}
    best_val = max(scores.values()) if higher_is_better else min(scores.values())
    best_m   = [m for m, v in scores.items() if v == best_val][0]

    champions = set()
    for m in available:
        p = paired_ttest(DATA[(m, ds)][metric], DATA[(best_m, ds)][metric])
        if p >= ALPHA:  # not significantly different from best
            champions.add(m)
    champions.add(best_m)  # always include the best
    return champions

print("Helper functions defined.")

In [ ]:
# ── Bold decisions: best overall (all methods) ────────────────────────────────
print("=== BOLD = statistical champion set (best overall, p<0.05) ===")
print()

bold = {}  # (method, ds, metric) -> True/False

for ds in DATASETS:
    for metric, hib in [('ci', True), ('ibs', False)]:
        champs = best_methods(ALL_METHODS, ds, metric, hib)
        means  = {m: np.mean(DATA[(m, ds)][metric]) for m in ALL_METHODS if (m, ds) in DATA}
        best_val = max(means.values()) if hib else min(means.values())
        label = f"{metric.upper()} ({'↑' if hib else '↓'})"
        print(f"{ds:10s}  {label}: best={best_val:.3f}  champions={sorted(champs)}")
        for m in ALL_METHODS:
            bold[(m, ds, metric)] = (m in champs)

In [ ]:
# ── Underline decisions: best zero-shot ───────────────────────────────────────
print("=== UNDERLINE = statistical champion set (best zero-shot, p<0.05) ===")
print()

underline = {}

for ds in DATASETS:
    for metric, hib in [('ci', True), ('ibs', False)]:
        champs = best_methods(ZERO_SHOT, ds, metric, hib)
        means  = {m: np.mean(DATA[(m, ds)][metric]) for m in ZERO_SHOT if (m, ds) in DATA}
        best_val = max(means.values()) if hib else min(means.values())
        label = f"{metric.upper()} ({'↑' if hib else '↓'})"
        print(f"{ds:10s}  {label}: best={best_val:.3f}  champions={sorted(champs)}")
        for m in ZERO_SHOT:
            underline[(m, ds, metric)] = (m in champs)

In [ ]:
# ── TabPFN vs TabICL pairwise comparison ──────────────────────────────────────
print("=== TabPFN vs TabICL: paired t-test (p-values, two-sided) ===")
print("  p < 0.05 → significantly different  |  direction shown as PFN>ICL or ICL>PFN")
print()

pairs = [
    ('TabSA-CCA (PFN)', 'TabSA-CCA (ICL)', 'CCA'),
    ('TabSA-Bin (PFN)', 'TabSA-Bin (ICL)', 'Bin'),
    ('TabSA-PO (PFN)',  'TabSA-PO (ICL)',  'PO'),
    ('TabSA-BJ (PFN)',  'TabSA-BJ (ICL)',  'BJ'),
]

rows = []
for pfn, icl, variant in pairs:
    for ds in DATASETS:
        if (pfn, ds) not in DATA or (icl, ds) not in DATA:
            continue
        for metric, hib in [('ci', True), ('ibs', False)]:
            a = np.array(DATA[(pfn, ds)][metric])
            b = np.array(DATA[(icl, ds)][metric])
            p = paired_ttest(a, b)
            diff = np.mean(a) - np.mean(b)
            if hib:
                winner = 'PFN' if diff > 0 else 'ICL'
            else:
                winner = 'PFN' if diff < 0 else 'ICL'
            sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            rows.append({'Variant': variant, 'Dataset': ds, 'Metric': metric.upper(),
                         'PFN_mean': np.mean(a), 'ICL_mean': np.mean(b),
                         'p_value': p, 'sig': sig, 'winner': winner if sig != 'ns' else '='})

cmp_df = pd.DataFrame(rows)
print(cmp_df.pivot_table(index=['Variant','Metric'], columns='Dataset',
                          values='sig', aggfunc='first')[DATASETS].to_string())
print()
print("Winner (where significant):")
print(cmp_df[cmp_df['sig'] != 'ns'][['Variant','Dataset','Metric','PFN_mean','ICL_mean','p_value','winner']].round(4).to_string(index=False))

In [ ]:
# ── Full table markup summary ─────────────────────────────────────────────────
print("=== TABLE MARKUP SUMMARY ===")
print("B=bold (best overall champion), U=underline (best zero-shot champion), BU=both")
print()

for ds in DATASETS:
    print(f"── {ds.upper()} ──")
    for m in ALL_METHODS:
        if (m, ds) not in DATA:
            continue
        tags_ci  = ('B' if bold.get((m,ds,'ci'),  False) else '') + \
                   ('U' if underline.get((m,ds,'ci'),  False) else '')
        tags_ibs = ('B' if bold.get((m,ds,'ibs'), False) else '') + \
                   ('U' if underline.get((m,ds,'ibs'), False) else '')
        ci_mean  = np.mean(DATA[(m,ds)]['ci'])
        ibs_mean = np.mean(DATA[(m,ds)]['ibs'])
        print(f"  {m:25s}  CI={ci_mean:.3f}[{tags_ci:2s}]  IBS={ibs_mean:.3f}[{tags_ibs:2s}]")
    print()